# 02 · Regla tradicional (caso base) y partición

**TFM:** Predicción temprana del abandono con datos de matrícula y modelos de ensamble frente a reglas tradicionales.

**Propósito.** Fijar la partición entrenamiento / prueba y calcular el desempeño de la regla tradicional, que es el caso base del piloto. La regla no se ajusta a los datos: no tiene parámetros que aprender, así que el conjunto de entrenamiento no se usa aquí. Se evalúa solo sobre prueba.

**Regla aprobada:** se marca en riesgo a quien aprobó menos de la mitad de las unidades curriculares matriculadas en el primer semestre. Los estudiantes sin unidades matriculadas (180, todos en la titulación 171) se marcan en riesgo y se reportan aparte. La regla no usa el segundo semestre: en E3 los modelos se comparan contra esta misma regla.

## 0. Configuración

In [1]:
# 0.1 Librerías, rutas y semilla
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

SEED = 42
N_BOOT = 2000
UMBRAL = 0.5          # proporción mínima de unidades aprobadas

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "README.md").exists())
DATA = ROOT / "TFM2" / "Datos" / "predict+students+dropout+and+academic+success" / "data.csv"
RESULTADOS = ROOT / "resultados"
RESULTADOS.mkdir(exist_ok=True)

pd.set_option("display.width", 140)
print(DATA.relative_to(ROOT))

TFM2/Datos/predict+students+dropout+and+academic+success/data.csv


## 1. Carga y objetivo binario

Misma carga que en `01_eda_dataset`: separador `;`, nombres sin espacios sobrantes, objetivo = abandono (1) frente a graduado o matriculado (0).

In [2]:
# 1.1 Carga
df = pd.read_csv(DATA, sep=";", encoding="utf-8-sig")
df.columns = df.columns.str.strip()
df["dropout"] = (df["Target"] == "Dropout").astype(int)
assert df.shape == (4424, 38)
print(df.shape, "| abandono:", int(df["dropout"].sum()), f"({df['dropout'].mean():.1%})")

(4424, 38) | abandono: 1421 (32.1%)


## 2. Partición entrenamiento / prueba

Estratificada por abandono, 80/20, semilla fija. Se define una sola vez en este cuaderno. Los modelos de `03_modelos` usarán exactamente estos mismos índices.

In [3]:
# 2.1 Partición
idx_train, idx_test = train_test_split(
    df.index, test_size=0.20, stratify=df["dropout"], random_state=SEED
)
train = df.loc[idx_train]
test = df.loc[idx_test]

resumen = pd.DataFrame({
    "n": [len(train), len(test)],
    "abandono": [int(train["dropout"].sum()), int(test["dropout"].sum())],
    "% abandono": [round(100 * train["dropout"].mean(), 1), round(100 * test["dropout"].mean(), 1)],
}, index=["entrenamiento", "prueba"])
print("Solapamiento entrenamiento/prueba:", len(set(idx_train) & set(idx_test)))
resumen

Solapamiento entrenamiento/prueba: 0


,n,abandono,% abandono
entrenamiento,3539,1137,32.1
prueba,885,284,32.1


## 3. Definición de la regla

La regla usa solo el primer semestre. Los casos con cero unidades matriculadas no tienen proporción definida: se marcan en riesgo y se identifican en una columna aparte.

In [4]:
# 3.1 Columnas del primer semestre
A1, M1 = "Curricular units 1st sem (approved)", "Curricular units 1st sem (enrolled)"

In [5]:
# 3.2 Regla del primer semestre
def regla_e2(d):
    sin_unidades = d[M1] == 0
    proporcion = d[A1] / d[M1].where(~sin_unidades)
    return ((proporcion < UMBRAL) | sin_unidades).astype(int)

sin_unidades = df[M1] == 0
df["regla"] = regla_e2(df)
df["sin_unidades"] = sin_unidades.astype(int)
train = df.loc[idx_train]
test = df.loc[idx_test]

pd.DataFrame({
    "% señalados (total)": [round(100 * df["regla"].mean(), 1)],
    "% señalados (sin los 180)": [round(100 * df.loc[~sin_unidades, "regla"].mean(), 1)],
}, index=["Regla, primer semestre"])

,% señalados (total),% señalados (sin los 180)
"Regla, primer semestre",23.2,20.0


## 4. Métricas en el conjunto de prueba

Métrica principal: *recall* de la clase abandono. Se reportan también precisión, F1 y la proporción de estudiantes señalados, porque la comparación con los modelos se hace con esa proporción fijada.

In [6]:
# 4.1 Función de métricas
def metricas(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "n": len(y_true),
        "abandonos": int(np.sum(y_true)),
        "señalados": int(np.sum(y_pred)),
        "% señalados": float(round(100 * np.mean(y_pred), 1)),
        "recall": round(recall_score(y_true, y_pred), 3),
        "precisión": round(precision_score(y_true, y_pred, zero_division=0), 3),
        "F1": round(f1_score(y_true, y_pred), 3),
        "falsos positivos": int(fp),
        "tasa falsos positivos": float(round(fp / (fp + tn), 3)),
    }

In [7]:
# 4.2 Regla en prueba
resultados = pd.DataFrame({"Regla, primer semestre": metricas(test["dropout"], test["regla"])}).T
resultados

,n,abandonos,señalados,% señalados,recall,precisión,F1,falsos positivos,tasa falsos positivos
"Regla, primer semestre",885.0,284.0,203.0,22.9,0.581,0.813,0.678,38.0,0.063


In [8]:
# 4.3 Casos sin unidades matriculadas, aparte
prueba_sin = test[test["sin_unidades"] == 1]
prueba_con = test[test["sin_unidades"] == 0]
aparte = pd.DataFrame({
    "n en prueba": [len(prueba_sin), len(prueba_con)],
    "abandono": [int(prueba_sin["dropout"].sum()), int(prueba_con["dropout"].sum())],
    "% abandono": [round(100 * prueba_sin["dropout"].mean(), 1), round(100 * prueba_con["dropout"].mean(), 1)],
}, index=["sin unidades matriculadas", "con unidades matriculadas"])
print("Regla sin los casos sin unidades:")
print(metricas(prueba_con["dropout"], prueba_con["regla"]))
aparte

Regla sin los casos sin unidades:
{'n': 851, 'abandonos': 263, 'señalados': 169, '% señalados': 19.9, 'recall': 0.548, 'precisión': 0.852, 'F1': 0.667, 'falsos positivos': 25, 'tasa falsos positivos': 0.043}


,n en prueba,abandono,% abandono
sin unidades matriculadas,34,21,61.8
con unidades matriculadas,851,263,30.9


## 5. Incertidumbre: intervalo bootstrap del recall

Remuestreo de la prueba con reemplazo, 2.000 repeticiones. Este intervalo es el que se usará en `03_modelos` para decidir si la diferencia de *recall* con el modelo es «no inferior en más de 5 puntos».

In [9]:
# 5.1 Bootstrap del recall de la regla
rng = np.random.default_rng(SEED)
y = test["dropout"].to_numpy()
pred = test["regla"].to_numpy()
boot = []
for _ in range(N_BOOT):
    i = rng.integers(0, len(y), len(y))
    boot.append(recall_score(y[i], pred[i], zero_division=0))

ic = pd.DataFrame({
    "recall": [recall_score(y, pred)],
    "IC 95 % inf.": [np.percentile(boot, 2.5)],
    "IC 95 % sup.": [np.percentile(boot, 97.5)],
}, index=["Regla, primer semestre"]).round(3)
ic

,recall,IC 95 % inf.,IC 95 % sup.
"Regla, primer semestre",0.581,0.524,0.64


## 6. Guardar resultados

Las métricas se guardan en `resultados/` para citarlas en los capítulos 4 y 5. Las predicciones de la regla quedan en la partición para que `03_modelos` las compare.

In [10]:
# 6.1 Exportar
resultados.to_csv(RESULTADOS / "regla_base_prueba.csv", encoding="utf-8")
ic.to_csv(RESULTADOS / "regla_base_bootstrap.csv", encoding="utf-8")
aparte.to_csv(RESULTADOS / "regla_base_sin_unidades.csv", encoding="utf-8")
pd.Series(idx_train, name="idx").to_csv(RESULTADOS / "idx_train.csv", index=False)
pd.Series(idx_test, name="idx").to_csv(RESULTADOS / "idx_test.csv", index=False)
print(sorted(p.name for p in RESULTADOS.iterdir()))

['idx_test.csv', 'idx_train.csv', 'regla_base_bootstrap.csv', 'regla_base_prueba.csv', 'regla_base_sin_unidades.csv']


## 7. Lectura de resultados

Se completa tras ejecutar el cuaderno, con las cifras de las celdas 4.2, 4.3 y 5.1. Las interpretaciones concretas se redactan en el capítulo 5 y no aquí.